# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    BaggingRegressor, 
    RandomForestRegressor, 
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

# Progress Tracking
from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [2]:
# =============================================================================
# Prelude: Reproduce Milestone 1 cleaning pipeline (Parts 3A-3E) and prepare
# the dataset for modeling. No feature engineering yet — that comes in Part 2.
# =============================================================================

# --- Load raw data ---
df_raw = pd.read_csv("zillow_dataset.csv")
print(f"Raw dataset: {df_raw.shape}")

# --- 3A: Drop features unsuitable for regression ---
drop_unsuitable = [
    'parcelid', 'assessmentyear', 'rawcensustractandblock',
    'censustractandblock', 'propertycountylandusecode',
    'propertyzoningdesc', 'regionidcounty'
]
df1 = df_raw.drop(columns=[c for c in drop_unsuitable if c in df_raw.columns])
print(f"After dropping unsuitable features: {df1.shape}")

# --- 3B: Drop features with >70% missing ---
missing_pct = (df1.isnull().sum() / len(df1) * 100)
high_missing = missing_pct[missing_pct > 70.0].index.tolist()
df2 = df1.drop(columns=high_missing)
print(f"After dropping high-missing features: {df2.shape}")

# --- 3C: Drop problematic samples ---
target = 'taxvaluedollarcnt'

# Drop rows with null target
df3 = df2.dropna(subset=[target])

# Drop rows with >50% missing features
min_non_null = int(df3.shape[1] * 0.5)
df3 = df3.dropna(thresh=min_non_null)

# Remove target outliers using IQR
Q1 = df3[target].quantile(0.25)
Q3 = df3[target].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
df3 = df3[(df3[target] >= lower) & (df3[target] <= upper)]
print(f"After dropping problematic samples: {df3.shape}")

# --- 3D: Impute remaining missing values ---
# Zero-fill (feature not present)
for col in ['garagecarcnt', 'garagetotalsqft']:
    if col in df3.columns:
        df3[col] = df3[col].fillna(0)

# Mode imputation for categorical/ID features
cat_cols = ['airconditioningtypeid', 'heatingorsystemtypeid', 'buildingqualitytypeid',
            'propertylandusetypeid', 'regionidcity', 'regionidneighborhood',
            'regionidzip', 'unitcnt', 'fips']
for col in cat_cols:
    if col in df3.columns:
        df3[col] = df3[col].fillna(df3[col].mode()[0])

# Median imputation for numerical features
num_cols = ['lotsizesquarefeet', 'finishedsquarefeet12', 'calculatedfinishedsquarefeet',
            'fullbathcnt', 'calculatedbathnbr', 'yearbuilt', 'roomcnt',
            'bathroomcnt', 'bedroomcnt', 'latitude', 'longitude']
for col in num_cols:
    if col in df3.columns:
        df3[col] = df3[col].fillna(df3[col].median())

print(f"After imputation — remaining nulls: {df3.isnull().sum().sum()}")

# --- 3E: One-hot encode categorical features ---
one_hot_cols = ['airconditioningtypeid', 'heatingorsystemtypeid', 'buildingqualitytypeid',
                'propertylandusetypeid', 'unitcnt', 'fips']
one_hot_present = [c for c in one_hot_cols if c in df3.columns]
df_cleaned = pd.get_dummies(df3, columns=one_hot_present, drop_first=True, dtype=int)
print(f"After encoding: {df_cleaned.shape}")

# =============================================================================
# Train/Test Split and Scaling
# =============================================================================
X = df_cleaned.drop(columns=[target])
y = df_cleaned[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

# Standardize features using only training data (no data leakage)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

print(f"Scaling complete. Features: {X_train_scaled.shape[1]}")
print(f"Target range: ${y_train.min():,.0f} - ${y_train.max():,.0f}")

Raw dataset: (77613, 55)
After dropping unsuitable features: (77613, 48)
After dropping high-missing features: (77613, 23)
After dropping problematic samples: (72332, 23)
After imputation — remaining nulls: 0
After encoding: (72332, 62)

Train set: (57865, 61), Test set: (14467, 61)
Scaling complete. Features: 61
Target range: $1,000 - $1,112,216


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [3]:
# =============================================================================
# Part 1: Baseline Setup — Repeated CV and results storage
# =============================================================================
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
baseline_results = {}

def run_baseline(name, model, X, y, cv):
    """Run repeated CV for a model and store results."""
    start = time.time()
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    baseline_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name}:")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} ± ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

In [4]:
# Baseline: Ridge Regression
# We originally intended to use plain Linear Regression (OLS), but it produced
# MAE in the trillions due to multicollinearity — near-duplicate features
# (e.g., finishedsquarefeet12 ≈ calculatedfinishedsquarefeet, bathroomcnt ≈
# calculatedbathnbr ≈ fullbathcnt) and high-cardinality categorical IDs treated
# as continuous numbers cause the OLS coefficient matrix to become near-singular.
# Ridge adds a small L2 penalty (alpha=1.0 by default) that stabilizes the
# coefficients without discarding any features — the standard fix for this issue.
run_baseline('Ridge Regression', Ridge(), X_train_scaled, y_train, cv)

Ridge Regression:
  CV MAE = $149,600.15 ± $1,229.54
  Time: 00:00:10



In [5]:
# Baseline: Random Forest (~8 min)
# commented out for now since it takes a bit to run and we already ran
# run_baseline('Random Forest', RandomForestRegressor(random_state=random_state), X_train_scaled, y_train, cv)
baseline_results['Random Forest'] = {'mean_mae': 136054.37, 'std_mae': 1070.61, 'time': 393}

In [6]:
# Baseline: HistGradientBoosting (~20 sec)
run_baseline('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state), X_train_scaled, y_train, cv)

HistGradientBoosting:
  CV MAE = $135,365.77 ± $1,142.28
  Time: 00:00:50



In [7]:
# Baseline Results Summary
baseline_df = pd.DataFrame(baseline_results).T
baseline_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']
baseline_df['Mean MAE ($)'] = baseline_df['Mean MAE ($)'].map('${:,.2f}'.format)
baseline_df['Std MAE ($)'] = baseline_df['Std MAE ($)'].map('${:,.2f}'.format)
baseline_df['Time (s)'] = baseline_df['Time (s)'].map(format_hms)
print('=== Baseline Results Summary ===')
display(baseline_df)

=== Baseline Results Summary ===


,Mean MAE ($),Std MAE ($),Time (s)
Ridge Regression,"$149,600.15","$1,229.54",00:00:10
Random Forest,"$136,054.37","$1,070.61",00:06:33
HistGradientBoosting,"$135,365.77","$1,142.28",00:00:50


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

**Best overall performance:** HistGradientBoosting achieved the lowest CV MAE at `$135,366` (`+-$1,142`), narrowly edging out Random Forest at `$136,054` (`+-$1,071`). Both tree-based ensembles substantially outperformed Ridge Regression (`$149,600` `+-$1,230`), which is expected — housing prices depend on nonlinear relationships and feature interactions (e.g., square footage × location) that linear models cannot capture.

**Most stable:** Random Forest had the lowest standard deviation (`$1,071`), making it the most consistent across CV folds. This aligns with its design — bagging reduces variance by averaging many decorrelated trees. HistGradientBoosting was nearly as stable (`$1,142`), while Ridge was only slightly higher (`$1,230`). All three models showed strong consistency.

**Overfitting / underfitting:** Ridge Regression shows clear signs of **underfitting** — its `~$14K` MAE gap to the tree models reflects the model's inability to capture nonlinear patterns in the data, not noise. The tree-based models do not show obvious signs of overfitting at this stage, given their tight standard deviations across folds, though this will be worth monitoring as we add engineered features. It's also worth noting that we originally attempted plain Linear Regression (OLS), which produced MAE in the trillions due to multicollinearity among near-duplicate features — Ridge's L2 regularization was the standard fix for this numerical instability.

**Computational note:** HistGradientBoosting was 24× faster than Random Forest (16 sec vs. 6:38) while achieving slightly better accuracy, thanks to its histogram-based binning approach. This speed advantage will be valuable during hyperparameter tuning in Part 4.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [8]:
# =============================================================================
# Part 2: Feature Engineering — Add new features from Milestone 1, Part 5
# We add these to the UNSCALED train/test sets, then re-scale.
# =============================================================================

def add_engineered_features(df):
    """Add engineered features based on Milestone 1 Part 5 analysis."""
    df_new = df.copy()
    
    # 1. Property age — older homes depreciate/appreciate differently
    df_new['property_age'] = 2016 - df_new['yearbuilt']
    
    # 2. Sqft per bedroom — captures spaciousness/room size
    df_new['sqft_per_bedroom'] = (
        df_new['calculatedfinishedsquarefeet'] / df_new['bedroomcnt'].replace(0, np.nan)
    ).fillna(0)
    
    # 3. Sqft x bathrooms — interaction capturing size + luxury signal
    df_new['sqft_x_bathrooms'] = (
        df_new['calculatedfinishedsquarefeet'] * df_new['bathroomcnt']
    )
    
    # 4. Log of lot size — reduces right-skew (showed improved correlation in M1)
    df_new['log_lotsizesquarefeet'] = np.log1p(df_new['lotsizesquarefeet'].clip(lower=0))
    
    return df_new

# Apply to both train and test (using unscaled data to preserve original values)
X_train_eng = add_engineered_features(X_train)
X_test_eng = add_engineered_features(X_test)

new_features = ['property_age', 'sqft_per_bedroom', 'sqft_x_bathrooms', 'log_lotsizesquarefeet']
print(f"Added {len(new_features)} engineered features: {new_features}")
print(f"Feature count: {X_train.shape[1]} -> {X_train_eng.shape[1]}")

# Re-scale with new features included
scaler_eng = StandardScaler()
X_train_eng_scaled = pd.DataFrame(
    scaler_eng.fit_transform(X_train_eng),
    columns=X_train_eng.columns,
    index=X_train_eng.index
)
X_test_eng_scaled = pd.DataFrame(
    scaler_eng.transform(X_test_eng),
    columns=X_test_eng.columns,
    index=X_test_eng.index
)
print("Scaling complete.")

Added 4 engineered features: ['property_age', 'sqft_per_bedroom', 'sqft_x_bathrooms', 'log_lotsizesquarefeet']
Feature count: 61 -> 65
Scaling complete.


In [9]:
# Part 2: CV setup and results storage
eng_results = {}

def run_eng(name, model, X, y, cv):
    """Run repeated CV and store results for Part 2."""
    start = time.time()
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    eng_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name}:")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} \u00b1 ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

In [10]:
# Part 2: Ridge Regression with engineered features
run_eng('Ridge Regression', Ridge(), X_train_eng_scaled, y_train, cv)

Ridge Regression:
  CV MAE = $149,470.40 ± $1,216.76
  Time: 00:00:00



In [11]:
# Part 2: Random Forest with engineered features
# commented out — already ran, hardcoding results to avoid ~9 min wait
# run_eng('Random Forest', RandomForestRegressor(n_estimators=100, random_state=random_state), X_train_eng_scaled, y_train, cv)
eng_results['Random Forest'] = {'mean_mae': 136505.27, 'std_mae': 1128.37, 'time': 565}

In [12]:
# Part 2: HistGradientBoosting with engineered features
run_eng('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state), X_train_eng_scaled, y_train, cv)

HistGradientBoosting:
  CV MAE = $135,424.51 ± $1,221.01
  Time: 00:00:41



In [13]:
# Part 2: Results comparison -- baseline vs. engineered features
eng_df = pd.DataFrame(eng_results).T
eng_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']

print('=== Part 2: Engineered Features Results ===')
for name in eng_results:
    new = eng_results[name]['mean_mae']
    old = baseline_results[name]['mean_mae']
    diff = new - old
    pct = (diff / old) * 100
    arrow = 'v' if diff < 0 else '^'
    print(f"  {name}: ${new:,.2f} (+/-${eng_results[name]['std_mae']:,.2f}) -- {arrow} ${abs(diff):,.2f} ({pct:+.2f}%)")

eng_df['Mean MAE ($)'] = eng_df['Mean MAE ($)'].map('${:,.2f}'.format)
eng_df['Std MAE ($)'] = eng_df['Std MAE ($)'].map('${:,.2f}'.format)
eng_df['Time (s)'] = eng_df['Time (s)'].map(format_hms)
display(eng_df)

=== Part 2: Engineered Features Results ===
  Ridge Regression: $149,470.40 (+/-$1,216.76) -- v $129.75 (-0.09%)
  Random Forest: $136,505.27 (+/-$1,128.37) -- ^ $450.90 (+0.33%)
  HistGradientBoosting: $135,424.51 (+/-$1,221.01) -- ^ $58.74 (+0.04%)


,Mean MAE ($),Std MAE ($),Time (s)
Ridge Regression,"$149,470.40","$1,216.76",00:00:00
Random Forest,"$136,505.27","$1,128.37",00:09:25
HistGradientBoosting,"$135,424.51","$1,221.01",00:00:41


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




**Notable improvement?** None of the three models showed meaningful improvement from the four engineered features. Ridge Regression saw the only (marginal) gain, dropping `$130` (-0.09%). Random Forest worsened slightly by `$451` (+0.33%), and HistGradientBoosting was essentially flat (`+$59`, +0.04%). These changes are all within the standard deviation of the CV scores, so they are not statistically significant.

**Which features helped, and for which models?** Ridge was the only model to benefit, likely because `sqft_x_bathrooms` and `property_age` give the linear model an explicit way to capture nonlinear relationships and interactions it can't discover on its own. For the tree-based models, these features are redundant — trees can already learn `sqft × bathrooms` through sequential splits and derive age implicitly from `yearbuilt`. Adding features the trees already "know" just adds noise dimensions that dilute the feature sampling (especially in Random Forest, which randomly selects feature subsets at each split).

**Why didn't the features help more?** The core issue is that our engineered features are all derived from existing columns — `property_age` is a linear transform of `yearbuilt`, `sqft_per_bedroom` is a ratio of two existing features, etc. They don't introduce genuinely new information. Features that would likely have more impact are ones the dataset lacks entirely, such as school district ratings, proximity to transit, crime statistics, or neighborhood-level price trends. The log transform of lot size also had minimal effect because the tree-based models handle skewed distributions natively through their splitting mechanism. Despite the modest results, we retain all four features going forward — they don't hurt, and feature selection in Part 3 will prune any that add noise.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [18]:
# =============================================================================
# Part 3: Feature Selection
# Strategy:
#   - Ridge: SelectKBest (f_regression) — appropriate for linear models
#   - Random Forest: feature importance from a fitted RF
#   - HistGBT: permutation importance from a fitted HGB
# We test multiple k values (top-k features) and pick the best for each model.
# =============================================================================

# --- Step 1: Get feature importances from each method ---

# F-regression scores (for Ridge)
from sklearn.feature_selection import f_regression
from sklearn.inspection import permutation_importance

f_scores, _ = f_regression(X_train_eng_scaled, y_train)
f_score_ranking = pd.Series(f_scores, index=X_train_eng_scaled.columns).sort_values(ascending=False)

# Random Forest feature importance (quick fit with 50 trees)
rf_temp = RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)
rf_temp.fit(X_train_eng_scaled, y_train)
rf_importance = pd.Series(rf_temp.feature_importances_, index=X_train_eng_scaled.columns).sort_values(ascending=False)

# HistGBT feature importance (using permutation importance)
hgb_temp = HistGradientBoostingRegressor(random_state=random_state)
hgb_temp.fit(X_train_eng_scaled, y_train)
perm_result = permutation_importance(hgb_temp, X_train_eng_scaled, y_train,
                                    n_repeats=5, random_state=random_state, n_jobs=-1)
hgb_importance = pd.Series(
  perm_result.importances_mean, index=X_train_eng_scaled.columns
).sort_values(ascending=False)

# Display top 15 features for each method
print("=== Top 15 Features by Method ===\n")
for name, ranking in [('F-score (Ridge)', f_score_ranking),
                     ('RF Importance', rf_importance),
                     ('HGB Importance', hgb_importance)]:
  print(f"{name}:")
  for feat, score in ranking.head(15).items():
      eng_marker = " *ENG*" if feat in new_features else ""
      print(f"  {feat}: {score:.4f}{eng_marker}")
  print()

=== Top 15 Features by Method ===

F-score (Ridge):
  finishedsquarefeet12: 17487.4292
  calculatedfinishedsquarefeet: 16988.8380
  sqft_x_bathrooms: 13906.9375 *ENG*
  calculatedbathnbr: 11272.6818
  bathroomcnt: 10611.3768
  fullbathcnt: 9524.3876
  sqft_per_bedroom: 6915.7931 *ENG*
  bedroomcnt: 3160.6194
  yearbuilt: 2944.2281
  property_age: 2944.2281 *ENG*
  garagetotalsqft: 2860.0458
  garagecarcnt: 2139.8382
  heatingorsystemtypeid_7.0: 1989.0904
  heatingorsystemtypeid_2.0: 1955.6759
  buildingqualitytypeid_9.0: 1706.4158

RF Importance:
  sqft_x_bathrooms: 0.2168 *ENG*
  latitude: 0.1536
  finishedsquarefeet12: 0.0950
  longitude: 0.0915
  sqft_per_bedroom: 0.0619 *ENG*
  regionidzip: 0.0528
  yearbuilt: 0.0480
  property_age: 0.0455 *ENG*
  calculatedfinishedsquarefeet: 0.0412
  log_lotsizesquarefeet: 0.0396 *ENG*
  lotsizesquarefeet: 0.0394
  regionidcity: 0.0244
  regionidneighborhood: 0.0224
  garagetotalsqft: 0.0159
  roomcnt: 0.0052

HGB Importance:
  latitude: 0.1679
 

In [19]:
# --- Step 2: Sweep top-k features for each model to find best subset ---
# We test k = 5, 10, 15, 20, 30, 45 using a quick 5-fold CV (no repeats) to find
# the best k, then do the full repeated CV on the winner.

k_values = [5, 10, 15, 20, 30, 45]
cv_quick = RepeatedKFold(n_splits=5, n_repeats=1, random_state=random_state)

selection_configs = {
    'Ridge Regression': {'ranking': f_score_ranking, 'model_fn': lambda: Ridge()},
    'Random Forest': {'ranking': rf_importance, 'model_fn': lambda: RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)},
    'HistGradientBoosting': {'ranking': hgb_importance, 'model_fn': lambda: HistGradientBoostingRegressor(random_state=random_state)},
}

best_k = {}  # store best k and features for each model

for model_name, config in selection_configs.items():
    print(f'{model_name}: sweeping k values...')
    best_mae = float('inf')
    for k in k_values:
        top_features = config['ranking'].head(k).index.tolist()
        X_subset = X_train_eng_scaled[top_features]
        scores = cross_val_score(
            config['model_fn'](), X_subset, y_train,
            cv=cv_quick, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        mae = (-scores).mean()
        marker = '  <-- best' if mae < best_mae else ''
        print(f'  k={k:2d}: MAE = ${mae:,.2f}{marker}')
        if mae < best_mae:
            best_mae = mae
            best_k[model_name] = {'k': k, 'features': top_features, 'mae_quick': mae}
    print(f'  Best: k={best_k[model_name]["k"]}\n')

# Show selected features for each model
for model_name, info in best_k.items():
    eng_count = len([f for f in info['features'] if f in new_features])
    print(f'{model_name} (k={info["k"]}): {eng_count} engineered features retained')
    print(f'  Features: {info["features"]}')
    print()

Ridge Regression: sweeping k values...
  k= 5: MAE = $162,761.85  <-- best
  k=10: MAE = $159,949.18  <-- best
  k=15: MAE = $157,516.13  <-- best
  k=20: MAE = $152,316.69  <-- best
  k=30: MAE = $151,179.41  <-- best
  k=45: MAE = $150,845.52  <-- best
  Best: k=45

Random Forest: sweeping k values...
  k= 5: MAE = $141,823.60  <-- best
  k=10: MAE = $137,326.76  <-- best
  k=15: MAE = $137,029.15  <-- best
  k=20: MAE = $136,880.04  <-- best
  k=30: MAE = $136,793.01  <-- best
  k=45: MAE = $136,851.50
  Best: k=30

HistGradientBoosting: sweeping k values...
  k= 5: MAE = $136,133.21  <-- best
  k=10: MAE = $135,767.40  <-- best
  k=15: MAE = $135,351.53  <-- best
  k=20: MAE = $135,416.53
  k=30: MAE = $135,459.80
  k=45: MAE = $135,256.08  <-- best
  Best: k=45

Ridge Regression (k=45): 4 engineered features retained
  Features: ['finishedsquarefeet12', 'calculatedfinishedsquarefeet', 'sqft_x_bathrooms', 'calculatedbathnbr', 'bathroomcnt', 'fullbathcnt', 'sqft_per_bedroom', 'bedro

In [20]:
# Part 3: Ridge with selected features (full repeated CV)
sel_results = {}

def run_selected(name, model, features, y, cv):
    """Run repeated CV on selected feature subset."""
    start = time.time()
    X_sub = X_train_eng_scaled[features]
    scores = cross_val_score(model, X_sub, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    sel_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name} (k={len(features)}):")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} +/- ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

run_selected('Ridge Regression', Ridge(), best_k['Ridge Regression']['features'], y_train, cv)

Ridge Regression (k=45):
  CV MAE = $150,861.95 +/- $1,230.40
  Time: 00:00:00



In [21]:
# Part 3: Random Forest with selected features (full repeated CV)
run_selected('Random Forest', RandomForestRegressor(random_state=random_state, n_jobs=-1),
             best_k['Random Forest']['features'], y_train, cv)

Random Forest (k=30):
  CV MAE = $136,522.73 +/- $1,123.94
  Time: 00:07:37



In [25]:
# Part 3: Random Forest with selected features (full repeated CV)
# commented out — already ran, hardcoding results to avoid ~8 min wait
# run_selected('Random Forest', RandomForestRegressor(random_state=random_state, n_jobs=-1),
#              best_k['Random Forest']['features'], y_train, cv)
sel_results['Random Forest'] = {'mean_mae': 136522.73, 'std_mae': 1123.94, 'time': 457}

In [24]:
# Part 3: Results comparison -- feature selection vs. Part 2 (all engineered features)
print('=== Part 3: Feature Selection Results ===')
for name in sel_results:
    new = sel_results[name]['mean_mae']
    p2 = eng_results[name]['mean_mae']
    p1 = baseline_results[name]['mean_mae']
    diff_p2 = new - p2
    diff_p1 = new - p1
    k = best_k[name]['k']
    print(f"  {name} (k={k}): ${new:,.2f} (+/-${sel_results[name]['std_mae']:,.2f})")
    print(f"    vs Part 2 (65 features): {'+' if diff_p2 > 0 else ''}{diff_p2:,.2f}")
    print(f"    vs Part 1 (61 features): {'+' if diff_p1 > 0 else ''}{diff_p1:,.2f}")

sel_df = pd.DataFrame(sel_results).T
sel_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']
sel_df['k'] = [best_k[name]['k'] for name in sel_results]
sel_df['Mean MAE ($)'] = sel_df['Mean MAE ($)'].map('${:,.2f}'.format)
sel_df['Std MAE ($)'] = sel_df['Std MAE ($)'].map('${:,.2f}'.format)
sel_df['Time (s)'] = sel_df['Time (s)'].map(format_hms)
display(sel_df)

=== Part 3: Feature Selection Results ===
  Ridge Regression (k=45): $150,861.95 (+/-$1,230.40)
    vs Part 2 (65 features): +1,391.55
    vs Part 1 (61 features): +1,261.80
  Random Forest (k=30): $136,522.73 (+/-$1,123.94)
    vs Part 2 (65 features): +17.46
    vs Part 1 (61 features): +468.36
  HistGradientBoosting (k=45): $135,362.34 (+/-$1,214.62)
    vs Part 2 (65 features): -62.17
    vs Part 1 (61 features): -3.43


,Mean MAE ($),Std MAE ($),Time (s),k
Ridge Regression,"$150,861.95","$1,230.40",00:00:00,45
Random Forest,"$136,522.73","$1,123.94",00:07:37,30
HistGradientBoosting,"$135,362.34","$1,214.62",00:00:12,45


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


Did performance improve after reducing features? Only HistGradientBoosting saw improvement, and it was marginal — a `$62` reduction in MAE (from `$135,425` to `$135,362`) when selecting the top 45 features by permutation importance. Random Forest was essentially flat at k=30 (`+$17` vs Part 2), while Ridge actually worsened by `$1,392` at k=45. The takeaway is that feature selection did not yield significant gains for any model, suggesting the original feature set was already reasonably well-curated from Milestone 1's cleaning pipeline.

Consistently retained features across models: Several features appeared in all three models' selected subsets: `calculatedfinishedsquarefeet`, `finishedsquarefeet12`, `bathroomcnt`, `yearbuilt`, `garagetotalsqft`, `bedroomcnt`, `latitude`, and the one-hot encoded building quality and heating system indicators. The tree models additionally valued `longitude`, `regionidzip`, and `regionidcity` — geographic features that capture neighborhood-level price variation through nonlinear splits, which F-regression (used for Ridge) cannot detect.

Were engineered features selected? Yes — `sqft_x_bathrooms` and `sqft_per_bedroom` were retained by all three models, confirming they carry useful signal. `property_age` was retained by Ridge and RF (and HGB at k=45). `log_lotsizesquarefeet` was only retained by RF. Notably, `sqft_x_bathrooms` ranked as the single most important feature for Random Forest (importance = 0.217), even above latitude — this interaction feature explicitly encodes "large homes with many bathrooms are expensive," a relationship trees can learn but benefit from having pre-computed. Despite this, the overall impact on MAE was minimal because the tree models were already approximating these relationships through splits on the constituent features.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [15]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [16]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here